# Setup

In [ ]:
%pip install -U upstash-vector

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from collections import deque
import time
import json
from pathlib import Path
import re

# Crawler

## Mapeamento

In [2]:
RECRAWL = False
BASE_URL = "https://ifrs.edu.br/canoas/"
DRIVE_DOWNLOAD = "https://drive.google.com/uc?export=download&id="
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

# extensoes ignoradas
IGNORED_EXTENSIONS = (".jpg", ".jpeg", ".png", ".gif", ".webp",
                      ".xlsx", ".xls", ".zip", ".ods", ".doc", ".ppt", ".docx")

# padrão de anos antigos agora vai até 2024
OLD_YEAR_PATTERN = re.compile(r"(201[0-9]|202[0-4])")

In [3]:
# define funções auxiliares
def is_valid_page(url):
    return url.startswith(BASE_URL)

def is_pdf_by_extension(url):
    return url.lower().endswith(".pdf")

def is_drive_link(url):
    return "drive.google.com/file/d/" in url

def extract_drive_id(url):
    # extrai o ID do arquivo do link do Drive
    parts = url.split("/file/d/")
    return parts[1].split("/")[0]

def build_download_url(url):
    # converte link do Drive para URL de download direto
    if is_drive_link(url):
        file_id = extract_drive_id(url)
        return DRIVE_DOWNLOAD + file_id
    return url

def should_ignore(url):
    url_lower = url.lower()
    
    # # extensões não úteis
    if any(url_lower.endswith(ext) for ext in IGNORED_EXTENSIONS):
        return True
    
    # # anos antigos no slug
    if OLD_YEAR_PATTERN.search(url):
        return True
    
    # # # páginas de paginação e listagem
    # if "/page/" in url or "/category/" in url:
    #     return True
    
    # # páginas temáticas sem valor para estudante
    if any(k in url_lower for k in ["/10anos/", "covid", "apnps", "retornoseguro", "vacina"]):
        return True
    
    # # páginas e keywords de baixo valor
    if "/paginateste/" in url or "/10anos/depoimentos/" in url:
        return True
    
    keywords = ["balanco-", "balanco_", "demonstracao-", "demonstracao_",
                "ata-concamp", "ptd-", "ptd_", "plano-de-trabalho"]
    if any(k in url_lower for k in keywords):
        return True
    
    return False

In [4]:
if RECRAWL:
    pages_found = []
    pdfs_found = []
    visited = set()
    queue = deque([BASE_URL])
    queued = set([BASE_URL])
else:
    pages_path = Path("../data/raw/pages.json")
    pdfs_path = Path("../data/raw/pdfs.json")

    if pages_path.exists():
        with open(pages_path, "r", encoding="utf-8") as f:
            pages_found = json.load(f)
    else:
        pages_found = []

    if pdfs_path.exists():
        with open(pdfs_path, "r", encoding="utf-8") as f:
            pdfs_found = json.load(f)
    else:
        pdfs_found = []

    already_known = set(pages_found) | {p["url"] for p in pdfs_found}
    visited = already_known.copy()
    queued = already_known.copy()
    queue = deque([BASE_URL])

    print(f"Dados existentes: {len(pages_found)} páginas, {len(pdfs_found)} PDFs")
    print(f"URLs já conhecidas ignoradas: {len(already_known)}")

Dados existentes: 3764 páginas, 631 PDFs
URLs já conhecidas ignoradas: 2660


In [5]:
# carrega whitelist de URLs forçadas
whitelist_path = Path("../data/info/whitelist.txt")
whitelist = set()

if whitelist_path.exists():
    with open(whitelist_path, "r", encoding="utf-8") as f:
        for line in f:
            url = line.strip()
            if url and not url.startswith("#"):
                whitelist.add(url)
                if url not in queued:
                    queue.append(url)
                    queued.add(url)
    print(f"Whitelist: {len(whitelist)} URLs forçadas")

Whitelist: 1 URLs forçadas


In [6]:
# inicia loop principal
while queue:
    url = queue.popleft()

    # pula se já visitado
    if url in visited:
        continue

    visited.add(url)
    total = len(visited) + len(queue)
    print(f"[{len(visited)}/{total}] Visitando: {url}")

    # faz a requisição
    try:
        response = requests.get(url, timeout=10, headers=HEADERS)
        response.raise_for_status()
        # detecta bloqueio por bot
        if "Radware" in response.text or "captcha" in response.text.lower():
            print(f"  BLOQUEADO: {url}")
            continue
    except Exception as e:
        print(f"  ERRO: {e}")
        continue

    # registra PDF e não extrai links dele
    if is_pdf_by_extension(url) or "application/pdf" in response.headers.get("Content-Type", ""):
        size_kb = len(response.content) / 1024
        pdfs_found.append({"url": url, "size_kb": round(size_kb, 2)})
        print(f"  PDF encontrado: {round(size_kb, 2)} KB")
        continue

    # só extrai links de páginas do domínio IFRS
    if not is_valid_page(url):
        continue

    # registra página HTML
    pages_found.append(url)

    # extrai links da página
    soup = BeautifulSoup(response.text, "html.parser")

    # páginas de listagem: extrai links mas não registra conteúdo
    is_listing = "/page/" in url or "/category/" in url
    if not is_listing:
        pages_found.append(url)

    novos = 0
    for tag in soup.find_all("a", href=True):
        href = tag["href"]
        full_url = urljoin(url, href).split("#")[0]

        # aplica filtros antes de enfileirar (whitelist bypassa should_ignore)
        if full_url not in whitelist and should_ignore(full_url):
            continue

        if full_url not in visited and full_url not in queued:
            if is_valid_page(full_url):
                queue.append(full_url)
                queued.add(full_url)
                novos += 1
            elif is_drive_link(full_url):
                # não enfileira, registra direto pra baixar depois com gdown
                download_url = build_download_url(full_url)
                if download_url not in queued:
                    pdfs_found.append({"url": download_url, "size_kb": 0})
                    queued.add(download_url)
                    novos += 1
            elif is_pdf_by_extension(full_url):
                queue.append(full_url)
                queued.add(full_url)
                novos += 1

    print(f"  {novos} novos links adicionados à fila")

    time.sleep(0.3)

# garante que a pasta existe
Path("../data/raw").mkdir(parents=True, exist_ok=True)

# salva páginas HTML
with open("../data/raw/pages.json", "w", encoding="utf-8") as f:
    json.dump(pages_found, f, ensure_ascii=False, indent=2)

# salva PDFs
with open("../data/raw/pdfs.json", "w", encoding="utf-8") as f:
    json.dump(pdfs_found, f, ensure_ascii=False, indent=2)

print(f"Salvo: {len(pages_found)} páginas e {len(pdfs_found)} PDFs")
print("Crawler finalizado.")

Salvo: 3764 páginas e 631 PDFs
Crawler finalizado.


In [7]:
total_pdf_size_kb = sum(p["size_kb"] for p in pdfs_found)

print("=== RESUMO DO CRAWLER ===")
print(f"Páginas HTML encontradas : {len(pages_found)}")
print(f"PDFs encontrados         : {len(pdfs_found)}")
print(f"Tamanho total dos PDFs   : {round(total_pdf_size_kb / 1024, 2)} MB")
print()
print("--- Primeiras 10 páginas ---")
for p in pages_found[:10]:
    print(p)
print()
print("--- Primeiros 10 PDFs ---")
for p in pdfs_found[:10]:
    print(f"{p['url']} ({p['size_kb']} KB)")

=== RESUMO DO CRAWLER ===
Páginas HTML encontradas : 3764
PDFs encontrados         : 631
Tamanho total dos PDFs   : 100.39 MB

--- Primeiras 10 páginas ---
https://ifrs.edu.br/canoas/
https://ifrs.edu.br/canoas/
https://ifrs.edu.br/canoas/acessibilidade/
https://ifrs.edu.br/canoas/acessibilidade/
https://ifrs.edu.br/canoas/sitemap/
https://ifrs.edu.br/canoas/sitemap/
https://ifrs.edu.br/canoas/cursos/
https://ifrs.edu.br/canoas/cursos/
https://ifrs.edu.br/canoas/contato/
https://ifrs.edu.br/canoas/contato/

--- Primeiros 10 PDFs ---
https://drive.google.com/uc?export=download&id=10ejt5bG3Si9j2utv3K40wkyFN0I73igf (0 KB)
https://drive.google.com/uc?export=download&id=1CK0P0IkJiLVtx__OTKE3PSLPYAV6ojXd (0 KB)
https://drive.google.com/uc?export=download&id=1E_2G9RHE_JsUE_6Txk55bdthmSm-8k24 (0 KB)
https://drive.google.com/uc?export=download&id=17RgbidQG8AE4C8TukKOUJuVb70PCfbWO (0 KB)
https://drive.google.com/uc?export=download&id=1rD7TIquNZtDEl0SZPGKMIcjakkO5bOm4 (0 KB)
https://drive.google.